### GOLD NOTEBOOK

This notebook builds the Gold layer of the Amazon sales dataset, transforming the Silver layer
into a fully normalized star schema ready for analytics in Power BI or Fabric.
Main steps performed in this notebook:
1. **Read Silver Layer**:
   - Load the cleaned and transformed Silver dataset as the starting point.
2. **Create Dimension Tables**:
   - **dim_customers**: unique CustomerID with CustomerName.
   - **dim_products**: unique ProductID with descriptive attributes (Category, Brand).
   - **dim_dates (dim_calendar)**: derived from OrderDate with Year, Month, Day, WeekOfYear, DayOfWeek, DayName, and IsWeekend.
   - **dim_payment_methods**: unique PaymentMethod with generated PaymentMethodID.
   - **dim_cities**: predefined list of 100 cities with generated CityID.
   - **dim_sellers**: unique SellerID from the fact dataset.
3. **Enrich Fact Table**:
   - Join fact table with PaymentMethodID.
   - Assign CityID cyclically for all orders.
   - Compute **TotalBill** = (UnitPrice * Quantity) + Tax + ShippingCost - Discount.
   - Drop descriptive columns now represented in dimensions, keeping only keys and numeric measures.
4. **Save Gold Tables**:
   - All tables (fact_sales + dimensions) are saved to the warehouse in Delta format.
   - Overwrite mode and schema updates ensure the tables are ready for analytics.

Result:
- A clean, normalized star schema with a fact table and all necessary dimensions.
- Ready for KPIs, aggregations, and interactive dashboards in Power BI.

This Gold layer completes the ETL pipeline from raw Silver data to analytics-ready Gold tables.


Import necessary libraries and functions for the Gold layer notebook.
- com.microsoft.spark.fabric: Fabric-specific Spark extensions for managing Lakehouse tables.
- pyspark.sql.functions: provides functions for column transformations, date operations,
  conditional logic, and generating unique IDs.
- pyspark.sql.window: allows creating window functions for row numbering or aggregations.
- SparkSession: entry point for Spark operations.
These imports prepare the notebook for building the star schema, creating fact and dimension tables,
and performing advanced transformations on the cleaned Silver dataset.

In [15]:
import com.microsoft.spark.fabric

from com.microsoft.spark.fabric.Constants import Constants
from pyspark.sql.functions import col, row_number, lpad, year, month, dayofmonth, date_format, weekofyear, dayofweek, when,first,lit, monotonically_increasing_id, first
from pyspark.sql.window import Window
from pyspark.sql import SparkSession

StatementMeta(, 1b98beed-c930-448a-9a5b-2a0681134f6e, 5, Finished, Available, Finished, False)

Load the cleaned Amazon dataset from the Silver layer into a Spark DataFrame.
This dataset has already been standardized: IDs cleaned, numeric and financial columns
cast to correct types, text fields trimmed, and dates formatted.
The DataFrame will be used to create the Gold layer, including fact and dimension tables
for the star schema.

In [16]:
df = spark.read.table("lakehouse_silver.DataFrame.amazon")

display(df)

StatementMeta(, 1b98beed-c930-448a-9a5b-2a0681134f6e, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b995abfd-1ad9-419e-a532-b7ecebb912f2)

In [17]:
fact_df = df

StatementMeta(, 1b98beed-c930-448a-9a5b-2a0681134f6e, 7, Finished, Available, Finished, False)

Create the Customer dimension table (dim_customers) from the fact dataset.
The table groups all records by CustomerID and keeps the first occurrence of CustomerName.
This ensures that each customer appears only once in the dimension table,
providing a clean reference for building relationships with the fact table.

In [19]:
dim_customers = fact_df.groupBy("CustomerID").agg(
    first("CustomerName").alias("CustomerName")
)

display(dim_customers)

StatementMeta(, 1b98beed-c930-448a-9a5b-2a0681134f6e, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 262a2dc7-f6db-427a-98df-aa4426c5cba1)

Perform a quality check on the Customer dimension table.
- total_rows counts all rows in dim_customers.
- distinct_ids counts the number of unique CustomerID values.
This ensures that each customer appears only once and helps detect potential duplicates or data issues.

In [21]:
total_rows = dim_customers.count()
distinct_ids = dim_customers.select("CustomerID").distinct().count()

print("Totale righe:", total_rows)
print("CustomerID distinti:", distinct_ids)

StatementMeta(, 1b98beed-c930-448a-9a5b-2a0681134f6e, 9, Finished, Available, Finished, False)

Totale righe: 43233
CustomerID distinti: 43233


Create the Product dimension table (dim_products) for the Gold layer.
1. Define a descriptive list of products with their Category and Brand.
2. Convert this list into a Spark DataFrame (dim_products_desc).
3. Join the descriptive DataFrame with the distinct ProductID and ProductName
   from the fact dataset to ensure the ProductID matches the fact table.
4. Select and reorder columns: ProductID, ProductName, Category, Brand.
This ensures a clean and complete Product dimension for the star schema,
ready to connect to the fact table for analytical queries.

In [23]:
spark = SparkSession.builder.getOrCreate()

# --- Lista de productos (información descriptiva) ---
products_list = [
    {"ProductName": "4K Monitor", "Category": "Electronics", "Brand": "BrightLux"},
    {"ProductName": "Action Camera", "Category": "Electronics", "Brand": "CoreTech"},
    {"ProductName": "Air Fryer", "Category": "Home & Kitchen", "Brand": "HomeEase"},
    {"ProductName": "Backpack", "Category": "Sports & Outdoors", "Brand": "UrbanStyle"},
    {"ProductName": "Bluetooth Speaker", "Category": "Electronics", "Brand": "Apex"},
    {"ProductName": "Board Game", "Category": "Toys & Games", "Brand": "KiddoFun"},
    {"ProductName": "Car Charger", "Category": "Electronics", "Brand": "NexPro"},
    {"ProductName": "Children's Book", "Category": "Books", "Brand": "ReadMore"},
    {"ProductName": "Cookware Set", "Category": "Home & Kitchen", "Brand": "HomeEase"},
    {"ProductName": "Desk Organizer", "Category": "Home & Kitchen", "Brand": "Zenith"},
    {"ProductName": "Desk Plant", "Category": "Home & Kitchen", "Brand": "HomeEase"},
    {"ProductName": "Dress Shirt", "Category": "Clothing", "Brand": "UrbanStyle"},
    {"ProductName": "Drone Mini", "Category": "Electronics", "Brand": "CoreTech"},
    {"ProductName": "Electric Kettle", "Category": "Home & Kitchen", "Brand": "HomeEase"},
    {"ProductName": "External HDD 2TB", "Category": "Electronics", "Brand": "NexPro"},
    {"ProductName": "Fitness Band", "Category": "Electronics", "Brand": "FitLife"},
    {"ProductName": "Gaming Mouse", "Category": "Electronics", "Brand": "CoreTech"},
    {"ProductName": "Graphic Tablet", "Category": "Electronics", "Brand": "Apex"},
    {"ProductName": "HDMI Cable 2m", "Category": "Electronics", "Brand": "NexPro"},
    {"ProductName": "Instant Pot", "Category": "Home & Kitchen", "Brand": "HomeEase"},
    {"ProductName": "Jeans", "Category": "Clothing", "Brand": "UrbanStyle"},
    {"ProductName": "Kids Toy Car", "Category": "Toys & Games", "Brand": "KiddoFun"},
    {"ProductName": "LED Desk Lamp", "Category": "Home & Kitchen", "Brand": "Zenith"},
    {"ProductName": "Laptop Sleeve", "Category": "Electronics", "Brand": "BrightLux"},
    {"ProductName": "Mechanical Keyboard", "Category": "Electronics", "Brand": "CoreTech"},
    {"ProductName": "Memory Card 128GB", "Category": "Electronics", "Brand": "NexPro"},
    {"ProductName": "Microphone", "Category": "Electronics", "Brand": "Apex"},
    {"ProductName": "Noise Cancelling Headphones", "Category": "Electronics", "Brand": "Apex"},
    {"ProductName": "Novel Bestseller", "Category": "Books", "Brand": "ReadMore"},
    {"ProductName": "Office Chair", "Category": "Home & Kitchen", "Brand": "Zenith"},
    {"ProductName": "Phone Tripod", "Category": "Electronics", "Brand": "NexPro"},
    {"ProductName": "Portable SSD 1TB", "Category": "Electronics", "Brand": "NexPro"},
    {"ProductName": "Power Bank 20000mAh", "Category": "Electronics", "Brand": "Apex"},
    {"ProductName": "Projector Mini", "Category": "Electronics", "Brand": "BrightLux"},
    {"ProductName": "Puzzle 1000pc", "Category": "Toys & Games", "Brand": "KiddoFun"},
    {"ProductName": "Router", "Category": "Electronics", "Brand": "NexPro"},
    {"ProductName": "Running Shoes", "Category": "Sports & Outdoors", "Brand": "UrbanStyle"},
    {"ProductName": "Smart Light Bulb", "Category": "Electronics", "Brand": "BrightLux"},
    {"ProductName": "Smartphone Case", "Category": "Electronics", "Brand": "Apex"},
    {"ProductName": "Smartwatch", "Category": "Electronics", "Brand": "FitLife"},
    {"ProductName": "Sunglasses", "Category": "Clothing", "Brand": "UrbanStyle"},
    {"ProductName": "T-Shirt", "Category": "Clothing", "Brand": "UrbanStyle"},
    {"ProductName": "USB-C Charger", "Category": "Electronics", "Brand": "NexPro"},
    {"ProductName": "Vacuum Cleaner", "Category": "Home & Kitchen", "Brand": "HomeEase"},
    {"ProductName": "Water Bottle", "Category": "Sports & Outdoors", "Brand": "FitLife"},
    {"ProductName": "Webcam Full HD", "Category": "Electronics", "Brand": "BrightLux"},
    {"ProductName": "Winter Jacket", "Category": "Clothing", "Brand": "UrbanStyle"},
    {"ProductName": "Wireless Charger", "Category": "Electronics", "Brand": "NexPro"},
    {"ProductName": "Wireless Earbuds", "Category": "Electronics", "Brand": "Apex"},
    {"ProductName": "Yoga Mat", "Category": "Sports & Outdoors", "Brand": "FitLife"}
]

# Crear DataFrame descriptivo
dim_products_desc = spark.createDataFrame(products_list)

# --- Tomamos ProductID de fact_df ---
# Suponemos que fact_df tiene columnas: ProductID (string) y ProductName
dim_products = dim_products_desc.join(
    fact_df.select("ProductName", "ProductID").distinct(),
    on="ProductName",
    how="left"
)

# Ahora ProductID queda **tal cual en fact_df**, no tocamos cast ni lpad
# Solo reordenamos columnas
dim_products = dim_products.select("ProductID", "ProductName", "Category", "Brand")

display(dim_products)

StatementMeta(, 1b98beed-c930-448a-9a5b-2a0681134f6e, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7c731175-1c39-4459-83a1-9cc5a1f2e490)

Create the Calendar dimension table (dim_calendar) for the Gold layer.
1. Select distinct OrderDate values from the fact dataset.
2. Derive date-related attributes:
   - Year, Month, MonthName, Day
   - WeekOfYear, DayOfWeek, DayName
   - IsWeekend (1 if Saturday or Sunday, 0 otherwise)
This dimension allows for time-based analysis and joins with the fact table
for aggregations, comparisons, and time intelligence calculations in Power BI.

In [24]:
dim_calendar = fact_df.select("OrderDate").distinct() \
    .withColumn("Year", year("OrderDate")) \
    .withColumn("Month", month("OrderDate")) \
    .withColumn("MonthName", date_format("OrderDate", "MMMM")) \
    .withColumn("Day", dayofmonth("OrderDate")) \
    .withColumn("WeekOfYear", weekofyear("OrderDate")) \
    .withColumn("DayOfWeek", dayofweek("OrderDate")) \
    .withColumn("DayName", date_format("OrderDate", "EEEE")) \
    .withColumn(
        "IsWeekend",
        when(dayofweek("OrderDate").isin(1,7), 1).otherwise(0)
    )

display(dim_calendar)

StatementMeta(, 1b98beed-c930-448a-9a5b-2a0681134f6e, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a54bd283-0e40-4d08-8057-47b7ffabf4c6)

Create the Payment Method dimension table (dim_payment_methods) for the Gold layer.
1. Select distinct PaymentMethod values from the fact dataset.
2. Generate a unique PaymentMethodID for each method using a row number over an ordered window.
   - The row number is converted to a string and left-padded with zeros to 3 digits (e.g., "001").
3. This produces a clean, consistent dimension table for joining with the fact table in the star schema.

In [25]:
window_spec = Window.orderBy("PaymentMethod")

dim_payment_methods = (
    fact_df.select("PaymentMethod")
    .distinct()
    .withColumn(
        "PaymentMethodID",
        lpad(row_number().over(window_spec).cast("string"), 3, "0")
    )
)

display(dim_payment_methods)

StatementMeta(, 1b98beed-c930-448a-9a5b-2a0681134f6e, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 051ab458-6a01-443a-9326-ba6a5cf06e63)

Create the City dimension table (dim_cities) for the Gold layer.
1. Define a list of 100 cities with their State and Country.
2. Convert this list into a Spark DataFrame.
3. Generate a unique CityID for each city using a row number over an arbitrary window,
   left-padded to 3 digits (e.g., "001").
4. Reorder columns: CityID, City, State, Country.
This produces a clean, standardized City dimension for the star schema,
ready to join with the fact table for location-based analysis in Power BI.

In [26]:
spark = SparkSession.builder.getOrCreate()

# Lista completa de 100 ciudades
cities_data = [
    {"City": "New York", "State": "NY", "Country": "United States"},
    {"City": "Los Angeles", "State": "CA", "Country": "United States"},
    {"City": "Chicago", "State": "IL", "Country": "United States"},
    {"City": "Houston", "State": "TX", "Country": "United States"},
    {"City": "Phoenix", "State": "AZ", "Country": "United States"},
    {"City": "Philadelphia", "State": "PA", "Country": "United States"},
    {"City": "San Antonio", "State": "TX", "Country": "United States"},
    {"City": "San Diego", "State": "CA", "Country": "United States"},
    {"City": "Dallas", "State": "TX", "Country": "United States"},
    {"City": "San Jose", "State": "CA", "Country": "United States"},
    {"City": "Austin", "State": "TX", "Country": "United States"},
    {"City": "Jacksonville", "State": "FL", "Country": "United States"},
    {"City": "Fort Worth", "State": "TX", "Country": "United States"},
    {"City": "Columbus", "State": "OH", "Country": "United States"},
    {"City": "Charlotte", "State": "NC", "Country": "United States"},
    {"City": "San Francisco", "State": "CA", "Country": "United States"},
    {"City": "Indianapolis", "State": "IN", "Country": "United States"},
    {"City": "Seattle", "State": "WA", "Country": "United States"},
    {"City": "Denver", "State": "CO", "Country": "United States"},
    {"City": "Washington", "State": "DC", "Country": "United States"},
    {"City": "Boston", "State": "MA", "Country": "United States"},
    {"City": "El Paso", "State": "TX", "Country": "United States"},
    {"City": "Nashville", "State": "TN", "Country": "United States"},
    {"City": "Detroit", "State": "MI", "Country": "United States"},
    {"City": "Oklahoma City", "State": "OK", "Country": "United States"},
    {"City": "Portland", "State": "OR", "Country": "United States"},
    {"City": "Las Vegas", "State": "NV", "Country": "United States"},
    {"City": "Memphis", "State": "TN", "Country": "United States"},
    {"City": "Louisville", "State": "KY", "Country": "United States"},
    {"City": "Baltimore", "State": "MD", "Country": "United States"},
    {"City": "Milwaukee", "State": "WI", "Country": "United States"},
    {"City": "Albuquerque", "State": "NM", "Country": "United States"},
    {"City": "Tucson", "State": "AZ", "Country": "United States"},
    {"City": "Fresno", "State": "CA", "Country": "United States"},
    {"City": "Sacramento", "State": "CA", "Country": "United States"},
    {"City": "Kansas City", "State": "MO", "Country": "United States"},
    {"City": "Mesa", "State": "AZ", "Country": "United States"},
    {"City": "Atlanta", "State": "GA", "Country": "United States"},
    {"City": "Omaha", "State": "NE", "Country": "United States"},
    {"City": "Colorado Springs", "State": "CO", "Country": "United States"},
    {"City": "Raleigh", "State": "NC", "Country": "United States"},
    {"City": "Miami", "State": "FL", "Country": "United States"},
    {"City": "London", "State": "ENG", "Country": "United Kingdom"},
    {"City": "Birmingham", "State": "WND", "Country": "United Kingdom"},
    {"City": "Manchester", "State": "GTR", "Country": "United Kingdom"},
    {"City": "Paris", "State": "IDF", "Country": "France"},
    {"City": "Lyon", "State": "ARA", "Country": "France"},
    {"City": "Marseille", "State": "PAC", "Country": "France"},
    {"City": "Berlin", "State": "BE", "Country": "Germany"},
    {"City": "Munich", "State": "BY", "Country": "Germany"},
    {"City": "Hamburg", "State": "HH", "Country": "Germany"},
    {"City": "Rome", "State": "Lazio", "Country": "Italy"},
    {"City": "Milan", "State": "Lombardy", "Country": "Italy"},
    {"City": "Madrid", "State": "MD", "Country": "Spain"},
    {"City": "Barcelona", "State": "CT", "Country": "Spain"},
    {"City": "Valencia", "State": "VC", "Country": "Spain"},
    {"City": "Toronto", "State": "ON", "Country": "Canada"},
    {"City": "Montreal", "State": "QC", "Country": "Canada"},
    {"City": "Vancouver", "State": "BC", "Country": "Canada"},
    {"City": "Calgary", "State": "AB", "Country": "Canada"},
    {"City": "Sydney", "State": "NSW", "Country": "Australia"},
    {"City": "Melbourne", "State": "VIC", "Country": "Australia"},
    {"City": "Brisbane", "State": "QLD", "Country": "Australia"},
    {"City": "Perth", "State": "WA", "Country": "Australia"},
    {"City": "Tokyo", "State": "TYO", "Country": "Japan"},
    {"City": "Osaka", "State": "OSA", "Country": "Japan"},
    {"City": "Nagoya", "State": "AIC", "Country": "Japan"},
    {"City": "Seoul", "State": "SNW", "Country": "South Korea"},
    {"City": "Busan", "State": "BSN", "Country": "South Korea"},
    {"City": "Beijing", "State": "BJ", "Country": "China"},
    {"City": "Shanghai", "State": "SH", "Country": "China"},
    {"City": "Hong Kong", "State": "HK", "Country": "China"},
    {"City": "Mumbai", "State": "MH", "Country": "India"},
    {"City": "Delhi", "State": "DL", "Country": "India"},
    {"City": "Bangalore", "State": "KA", "Country": "India"},
    {"City": "Mexico City", "State": "CDMX", "Country": "Mexico"},
    {"City": "Guadalajara", "State": "JAL", "Country": "Mexico"},
    {"City": "Monterrey", "State": "NL", "Country": "Mexico"},
    {"City": "Buenos Aires", "State": "CABA", "Country": "Argentina"},
    {"City": "Sao Paulo", "State": "SP", "Country": "Brazil"},
    {"City": "Rio de Janeiro", "State": "RJ", "Country": "Brazil"},
    {"City": "Cape Town", "State": "WC", "Country": "South Africa"},
    {"City": "Johannesburg", "State": "GP", "Country": "South Africa"},
    {"City": "Dubai", "State": "DU", "Country": "United Arab Emirates"},
    {"City": "Singapore", "State": "SG", "Country": "Singapore"},
    {"City": "Amsterdam", "State": "NH", "Country": "Netherlands"},
    {"City": "Brussels", "State": "BRU", "Country": "Belgium"},
    {"City": "Vienna", "State": "WIE", "Country": "Austria"},
    {"City": "Stockholm", "State": "AB", "Country": "Sweden"},
    {"City": "Oslo", "State": "03", "Country": "Norway"},
    {"City": "Copenhagen", "State": "84", "Country": "Denmark"},
    {"City": "Helsinki", "State": "18", "Country": "Finland"},
    {"City": "Dublin", "State": "L", "Country": "Ireland"},
    {"City": "Lisbon", "State": "11", "Country": "Portugal"},
    {"City": "Warsaw", "State": "14", "Country": "Poland"},
    {"City": "Prague", "State": "10", "Country": "Czech Republic"},
    {"City": "Istanbul", "State": "34", "Country": "Turkey"},
    {"City": "Bangkok", "State": "10", "Country": "Thailand"},
    {"City": "Jakarta", "State": "JK", "Country": "Indonesia"},
    {"City": "Kuala Lumpur", "State": "14", "Country": "Malaysia"}
]

# Crear DataFrame
df = spark.createDataFrame(cities_data)

# Window function para generar ID
window_spec = Window.orderBy(lit(1))  # Orden arbitrario
dim_cities = df.withColumn(
    "CityID",
    lpad(row_number().over(window_spec).cast("string"), 3, "0")
)

# Reordenar columnas
dim_cities = dim_cities.select("CityID", "City", "State", "Country")

# Mostrar resultado
display(dim_cities)

StatementMeta(, 1b98beed-c930-448a-9a5b-2a0681134f6e, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a5644671-cbd0-4be4-a8b9-d7a31b76cb41)

Create the Seller dimension table (dim_sellers) for the Gold layer.
1. Select distinct SellerID values from the fact dataset.
2. This produces a simple dimension table listing each seller only once.
The table can be used to join with the fact table for seller-level analysis in Power BI.

In [27]:
dim_sellers = fact_df.select("SellerID").distinct()

display(dim_sellers)

StatementMeta(, 1b98beed-c930-448a-9a5b-2a0681134f6e, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9ae4bbab-d3dc-401c-99a3-e07fac857442)

Enrich the fact table with PaymentMethodID from the Payment Method dimension.
1. Perform a left join between the fact table and dim_payment_methods on PaymentMethod.
2. Add PaymentMethodID to the fact table, ensuring that each record references the correct dimension key.
This step is crucial for establishing the foreign key relationship in the star schema.

In [28]:
fact_df = fact_df.join(
    dim_payment_methods.select("PaymentMethod", "PaymentMethodID"),
    on="PaymentMethod",
    how="left"
)

StatementMeta(, 1b98beed-c930-448a-9a5b-2a0681134f6e, 16, Finished, Available, Finished, False)

Assign CityID to the fact table for linking with the City dimension.
1. Create a sequential number for each row using a window with a monotonically increasing ID.
2. Map the sequential numbers to CityID values from 1 to 100 (modulo 100) to assign cities cyclically.
3. Convert the number to a zero-padded string of 3 digits (e.g., "001", "002").
4. Drop the temporary integer column after creating CityID.
This ensures each record in the fact table references a valid CityID from dim_cities.

In [29]:
window_spec = Window.orderBy(monotonically_increasing_id())

fact_df = fact_df.withColumn("CityID_int", row_number().over(window_spec))

fact_df = fact_df.withColumn("CityID_int", ((col("CityID_int") - 1) % 100) + 1)

fact_df = fact_df.withColumn("CityID", lpad(col("CityID_int").cast("string"), 3, "0"))

fact_df = fact_df.drop("CityID_int")

StatementMeta(, 1b98beed-c930-448a-9a5b-2a0681134f6e, 15, Finished, Available, Finished, False)

Create the final Gold fact table (fact_sales) for the star schema.
1. Calculate TotalBill for each order row:
   TotalBill = (UnitPrice * Quantity) + Tax + ShippingCost - Discount
2. Drop redundant or descriptive columns that are now represented in dimension tables:
   - TotalAmount, CustomerName, ProductName, Category, Brand, PaymentMethod, City, State, Country
3. Keep only keys (CustomerID, ProductID, SellerID, PaymentMethodID, CityID, OrderDate) 
   and numeric measures for analysis in Power BI.
This produces a clean, normalized fact table ready for aggregation and KPI calculations.

In [30]:
fact_sales = fact_df.withColumn(
    "TotalBill",
    (col("UnitPrice") * col("Quantity")) +
    col("Tax") +
    col("ShippingCost") -
    col("Discount")
).drop(
    "TotalAmount",
    "CustomerName",
    "ProductName",
    "Category",
    "Brand",
    "PaymentMethod",
    "City",
    "State",
    "Country"
)

display(fact_sales)

StatementMeta(, 1b98beed-c930-448a-9a5b-2a0681134f6e, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f37047cd-496f-478c-96b8-bfc141ad940d)

Save all Gold layer tables to the warehouse (Delta format) for consumption in Power BI / Fabric.
 FACT SALES table
   - fact_sales: contains all measures and foreign keys to dimensions.
 DIMENSIONS
   - dim_customers, dim_products, dim_dates, dim_payment_methods, dim_cities, dim_sellers
 Use Delta format with overwrite mode and overwriteSchema=True to ensure tables are up-to-date.
 Tables are written to the 'warehouse_gold.DataSets' schema, ready for analytics.
This completes the ETL pipeline from Silver to Gold layer with a clean star schema.

In [31]:
# ----------------------------------------
# FACT SALES
# ----------------------------------------
fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .synapsesql("warehouse_gold.DataSets.fact_sales")

# ----------------------------------------
# DIM CUSTOMERS
# ----------------------------------------
dim_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .synapsesql("warehouse_gold.DataSets.dim_customers")

# ----------------------------------------
# DIM PRODUCTS
# ----------------------------------------
dim_products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .synapsesql("warehouse_gold.DataSets.dim_products")

# ----------------------------------------
# DIM DATES
# ----------------------------------------
dim_calendar.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .synapsesql("warehouse_gold.DataSets.dim_dates")

# ----------------------------------------
# DIM PAYMENT METHODS
# ----------------------------------------
dim_payment_methods.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .synapsesql("warehouse_gold.DataSets.dim_payment_methods")

# ----------------------------------------
# DIM CITIES
# ----------------------------------------
dim_cities.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .synapsesql("warehouse_gold.DataSets.dim_cities")

# ----------------------------------------
# DIM SELLERS
# ----------------------------------------
dim_sellers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .synapsesql("warehouse_gold.DataSets.dim_sellers")

StatementMeta(, 1b98beed-c930-448a-9a5b-2a0681134f6e, 18, Finished, Available, Finished, False)